# M11 · Incremental-mode demo

**Goal:** show `incremental_flow()` running once — no-op if already caught up, otherwise processes a small delta. Demonstrates the 5-minute-cron contract.

**Exit criterion:** cell runs without error; `rows_loaded` is 0 if the warehouse is fully caught up.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — watermark snapshot before

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    before = pd.read_sql(text('SELECT * FROM warehouse.etl_watermark ORDER BY table_name'), conn)
before


## 3. Execute

In [ ]:
from accent_fleet.pipeline import incremental_flow
incremental_flow()


## 4. Inspect — watermark diff + latest run_log entry

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    after = pd.read_sql(text('SELECT * FROM warehouse.etl_watermark ORDER BY table_name'), conn)
    run  = pd.read_sql(text('SELECT * FROM warehouse.etl_run_log ORDER BY started_at DESC LIMIT 1'), conn)
display(after); display(run)
